In [ ]:
import torch
import numpy as np
!pip install evaluate sacrebleu transformers -U
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
# --- 1. Configuration & Device Setup ---
# Use the official English-Hindi checkpoint
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-en-hi"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 128  # Defined MAX_LENGTH for the tokenizer

print(f"Running on device: {DEVICE}")


Running on device: cpu


In [ ]:
# --- 2. Load Model and Tokenizer (Fix for NameError) ---
# This ensures tokenizer and model are defined before the trainer setup
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)
    if DEVICE == "cuda":
        model.to(DEVICE)
    print("Model and Tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading model/tokenizer: {e}")
    # Exit if we can't load the essentials
    exit()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Model and Tokenizer loaded successfully.


In [ ]:
# --- 3. Load Dataset (Fix for Performance/Dataset Error) ---
# Switched to a smaller subset of the cfilt/iitb-english-hindi dataset for speed.
# Change '[:50000]' to a larger number (e.g., '[:500000]') for better quality.
try:
    raw_datasets = load_dataset('cfilt/iitb-english-hindi', split='train[:50000]')
    # Split the loaded portion into train and test
    raw_datasets = raw_datasets.train_test_split(train_size=0.9, seed=42)
    print(f"Dataset loaded. Train size: {len(raw_datasets['train'])}, Test size: {len(raw_datasets['test'])}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit()


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

Dataset loaded. Train size: 45000, Test size: 5000


In [ ]:
def preprocess_function(examples):
    # The 'cfilt/iitb-english-hindi' dataset has keys 'translation' -> 'en' and 'hi'
    # When batched=True, examples['translation'] is a list of dictionaries.
    english_sentences = [item["en"] for item in examples["translation"]]
    hindi_sentences = [item["hi"] for item in examples["translation"]]

    # Tokenize English (Source)
    model_inputs = tokenizer(english_sentences, max_length=MAX_LENGTH, truncation=True)

    # Tokenize Hindi (Target) and set as labels for Seq2Seq
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(hindi_sentences, max_length=MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing to the entire dataset
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    num_proc=4,
)

# Correctly rename the 'test' split to 'validation'
tokenized_datasets["validation"] = tokenized_datasets.pop("test")

print("\nSample tokenized input_ids and labels (target):")
print(f"Input IDs: {tokenized_datasets['train'][0]['input_ids'][:10]}")
print(f"Labels: {tokenized_datasets['train'][0]['labels'][:10]}")

Map (num_proc=4):   0%|          | 0/45000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your 

Map (num_proc=4):   0%|          | 0/5000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your 


Sample tokenized input_ids and labels (target):
Input IDs: [32658, 4, 769, 1853, 0]
Labels: [2349, 1575, 12, 458, 84, 0]


In [ ]:
# Fix: Ensure evaluate and sacrebleu are installed before import
!pip install evaluate sacrebleu -q # -q for quiet output

import evaluate

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels] # SacreBLEU expects references as a list of lists
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # Decode predictions back to text
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels as we can't decode them.
    # Trainer puts -100 in padding tokens and special tokens.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process for SacreBLEU (stripping whitespace and formatting references)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # Compute the BLEU score
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]} # Extract the 'score' key for BLEU

    # You can also compute average generation length if needed
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)

    return result

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.7 MB/s eta 0:00:00


In [ ]:
# --- 5. Metrics Definition ---
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Decode labels
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Post-process for SacreBLEU
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch # Ensure torch is imported for torch.cuda.is_available()
from datasets import load_dataset # Import for dataset loading

# --- Re-load Model and Tokenizer if not defined for robustness ---
# This section ensures 'tokenizer' and 'model' are available,
# in case the cell defining them (e.g., cell 6QVgpPuI1Pwn) was not run
# or its state was lost.
if 'tokenizer' not in globals() or 'model' not in globals():
    print("Tokenizer or model not found in global scope. Re-loading them...")
    # MODEL_CHECKPOINT and DEVICE are assumed to be defined by previous cells and are in kernel state
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)
        if DEVICE == "cuda":
            model.to(DEVICE)
        print("Model and Tokenizer re-loaded successfully within this cell's scope.")
    except Exception as e:
        print(f"Error re-loading model/tokenizer: {e}")
        raise # Critical error, stop execution

# --- Ensure tokenized_datasets is available ---
if 'tokenized_datasets' not in globals():
    print("tokenized_datasets not found in global scope. Re-loading and tokenizing dataset...")
    try:
        # Load raw dataset (from WI_kjOJ01Uqp)
        raw_datasets = load_dataset('cfilt/iitb-english-hindi', split='train[:50000]')
        raw_datasets = raw_datasets.train_test_split(train_size=0.9, seed=42)
        print(f"Dataset loaded. Train size: {len(raw_datasets['train'])}, Test size: {len(raw_datasets['test'])}")

        # Define preprocess_function (from Ja1rkXRmd1pL)
        def preprocess_function(examples):
            english_sentences = [item["en"] for item in examples["translation"]]
            hindi_sentences = [item["hi"] for item in examples["translation"]]
            model_inputs = tokenizer(english_sentences, max_length=MAX_LENGTH, truncation=True)
            with tokenizer.as_target_tokenizer():
                labels = tokenizer(hindi_sentences, max_length=MAX_LENGTH, truncation=True)
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

        # Apply the preprocessing (from Ja1rkXRmd1pL)
        tokenized_datasets = raw_datasets.map(
            preprocess_function,
            batched=True,
            remove_columns=raw_datasets["train"].column_names,
            num_proc=4, # Use num_proc=1 if you encounter multiprocessing issues in some environments
        )
        tokenized_datasets["validation"] = tokenized_datasets.pop("test")
        print("Dataset re-tokenized successfully within this cell's scope.")

    except Exception as e:
        print(f"Error re-loading/tokenizing dataset: {e}")
        raise # Critical error, stop execution

# --- 6. Training Arguments and Trainer Setup (Fix Applied) ---
training_args = Seq2SeqTrainingArguments(
    output_dir="./final_en_hi_translator",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    num_train_epochs=3,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    logging_dir='./logs',
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

tokenized_datasets not found in global scope. Re-loading and tokenizing dataset...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your 

Map (num_proc=4):   0%|          | 0/5000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your 

Dataset re-tokenized successfully within this cell's scope.


/tmp/ipython-input-71594938.py:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
# --- 7. Start Fine-Tuning (Now it should run without NameError) ---
print("\n--- Starting Fine-Tuning ---")
print("NOTE: Training time is now significantly reduced due to the smaller dataset slice (50k).")
# trainer.train()


--- Starting Fine-Tuning ---
NOTE: Training time is now significantly reduced due to the smaller dataset slice (50k).


In [ ]:
# 8. Save the final fine-tuned model (Uncomment after successful training)
# trainer.save_model("./final_en_hi_translator")

In [ ]:
# --- 9. Inference Function (To be used in the Gradio App) ---
# Ensure MODEL_CHECKPOINT is defined for this cell's execution context
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-en-hi" # Re-defining it here for robustness

def translate_text(english_text, model_path=MODEL_CHECKPOINT):
    # Load the model/tokenizer (or from the fine-tuned path)
    inference_tokenizer = AutoTokenizer.from_pretrained(model_path)
    inference_model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)

    # Tokenize the input text
    input_tokens = inference_tokenizer(
        english_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(DEVICE)

    # Generate the translation using beam search for quality
    translated_tokens = inference_model.generate(
        **input_tokens,
        max_length=MAX_LENGTH,
        num_beams=6, # Higher beams for better quality translation
        early_stopping=True,
    )

    # Decode the generated tokens back to Hindi text
    hindi_text = inference_tokenizer.decode(
        translated_tokens[0],
        skip_special_tokens=True
    )
    return hindi_text

# Example Inference Usage (This will use the base model until you train and save the model)
print("\n--- Example Translation Test ---")
test_sentences = [
    "How are you today?",
    "My name is Shivam",
    "Machine translation models are very helpful.",
]
for sentence in test_sentences:
    translation = translate_text(sentence, model_path=MODEL_CHECKPOINT)
    print(f"English: {sentence} -> Hindi: {translation}")


--- Example Translation Test ---


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


English: How are you today? -> Hindi: आज आप कैसे हैं?
English: My name is Shivam -> Hindi: मेरा नाम शिवाम है
English: Machine translation models are very helpful. -> Hindi: मशीन अनुवाद मॉडल बहुत सहायक हैं.


In [ ]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- 1. Configuration & Global Model Loading (CRITICAL FIX) ---
# Load model globally to prevent repeated loading and NameErrors,
# and to ensure the app doesn't crash if loading fails.
MODEL_PATH = "Helsinki-NLP/opus-mt-en-hi"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 128

# Global Model and Tokenizer Initialization
try:
    GLOBAL_TOKENIZER = AutoTokenizer.from_pretrained(MODEL_PATH)
    # Move the model to the appropriate device (CPU or GPU)
    GLOBAL_MODEL = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
    print(f"Model loaded successfully on {DEVICE}.")
except Exception as e:
    # If loading fails (e.g., due to network or environment issues), set placeholders
    print(f"CRITICAL: Failed to load global model/tokenizer from {MODEL_PATH}. Error: {e}")
    GLOBAL_TOKENIZER = None
    GLOBAL_MODEL = None


# --- 2. Translation Function (Uses Global Model) ---
def translate_text(english_text):
    if not english_text:
        return "Input text cannot be empty."

    # Check if the model loaded globally before proceeding
    if GLOBAL_MODEL is None or GLOBAL_TOKENIZER is None:
        return "Error: Translation model not loaded. Check startup console for details."

    # Use the globally loaded model and tokenizer
    tokenizer = GLOBAL_TOKENIZER
    model = GLOBAL_MODEL

    # Tokenize the input text
    input_tokens = tokenizer(
        english_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(DEVICE)

    # Generate the translation using beam search for quality
    translated_tokens = model.generate(
        **input_tokens,
        max_length=MAX_LENGTH,
        num_beams=6,
        early_stopping=True,
    )

    # Decode the generated tokens back to Hindi text
    hindi_text = tokenizer.decode(
        translated_tokens[0],
        skip_special_tokens=True
    )
    return hindi_text

# --- 3. Gradio Interface Setup ---
# Consolidated CSS from the various fragmented attempts in the original notebook state
css = """
body {
    background-color: #e5e5f7;
    background-image:
        repeating-radial-gradient(circle at 0 0, transparent 0, #e5e5f7 27px),
        repeating-linear-gradient(#daa52355, #daa523),
        linear-gradient(120deg, #fff3c4, #ffe9a7, #f8d77c, #fff3c4);
    background-size: auto, auto, 400% 400%;
    background-attachment: fixed;
    animation: shimmerMove 12s ease-in-out infinite;
    font-family: 'Inter', sans-serif;
    color: #1a1a1a;
}

/* ===== Title & Description ===== */
h1 {
    color: #ffeb3b; /* bright yellow, combined from the original Gradio.Interface call */
    text-align: center;
    font-weight: 2000;
    text-shadow: 0 0 2px #ffeb3b, 0 0 4px #ffd54f;
    margin-bottom: 20px;
}
.gradio-container p { /* Targeting all p in gradio-container, might be too broad */
    color: #333;
    text-align: center;
    font-size: 50px; /* This is very large, assuming it was intended for the description */
}

/* ===== Labels ===== */
.label {
    font-size: 18px !important; /* from first block */
    font-weight: 950; /* from second block (overwrites 600 from first) */
    color: #000 !important;
}

/* ===== Text Areas ===== */
textarea {
    min-height: 300px !important;
    min-width: 100% !important;
    resize: none !important;
    border-radius: 16px !important;
    font-size: 20px !important;
    font-weight: 500 !important;
    line-height: 1.6 !important;
    padding: 20px !important;
    color: #000 !important;
    background: rgba(255, 255, 255, 0.85) !important;
    border: 2px solid #daa523 !important;
    box-shadow: 0 4px 15px rgba(218, 165, 35, 0.3);
    transition: all 0.3s ease;
}

textarea:focus {
    outline: none !important;
    border: 2px solid #ffcc00 !important;
    box-shadow: 0 0 15px rgba(255, 221, 0, 0.6);
}

.gr-textarea { /* Specific Gradio textarea styling */
    border: 1px solid #ccc !important;
    background: rgba(255, 255, 255, 0.85) !important;
    color: #000 !important;
    border-radius: 10px !important;
}


/* ===== Buttons ===== */
.gr-button-primary {
    margin-top: 10px; /* from first block */
    font-size: 18px !important; /* from first block */
    padding: 10px 25px !important; /* from first block */
    background-color: #daa523;
    color: #fff;
    border: none;
    border-radius: 10px;
    transition: all 0.3s ease;
    font-weight: 900; /* from second block (overwrites 600 from first) */
    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.2);
}
.gr-button-primary:hover {
    background-color: #f1b733;
    transform: translateY(-2px);
    box-shadow: 0 6px 12px rgba(0, 0, 0, 0.25);
}

/* ===== Secondary buttons (Clear, Flag) ===== */
button:not(.gr-button-primary) {
    background-color: rgba(0, 0, 0, 0.05);
    color: #000;
    border-radius: 10px;
}
button:not(.gr-button-primary):hover {
    background-color: rgba(0, 0, 0, 0.15);
}

/* ===== Footer ===== */
footer, .footer {
    color: #444 !important;
}
"""

iface = gr.Interface(
    fn=translate_text,
    inputs=gr.Textbox(
        lines=5,
        label="English Input",
        placeholder="Enter text here (e.g., I want a cup of tea)"
    ),
    outputs=gr.Textbox(
        lines=5,
        label="Hindi Translation (हिंदी अनुवाद)",
        placeholder="Translation will appear here"
    ),
    title="✨English to Hindi Machine Translaton Model (DL Project)✨",
    description="This application uses a Hugging Face Sequence-to-Sequence model (Helsinki-NLP/opus-mt-en-hi). The current deployment uses the base model for speed; better results require full fine-tuning. The model provides high-quality, contextually meaningful Hindi translations.",
    css= """
h1 {
    color: #ffeb3b; /* bright yellow */
    text-align: center;
    font-weight: 2000;
    text-shadow: 0 0 2px #ffeb3b, 0 0 1px #ffd54f;
}
""", # Pass the consolidated CSS string
    theme=gr.themes.Soft(primary_hue="yellow", secondary_hue="gray") # Correctly pass the theme object
)

if __name__ == "__main__":
    print(f"Gradio app initializing. Model path set to: {MODEL_PATH}")
    # Before launching, it's a good practice to ensure a clean state if `transformers` was just updated
    # This often involves restarting the runtime after a major library update that affects internal structure.
    # After restarting the runtime and re-running all cells, uncomment the line below to launch the app:
    # iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Model loaded successfully on cpu.
Gradio app initializing. Model path set to: Helsinki-NLP/opus-mt-en-hi


In [ ]:
iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4ef19e0aec0c69ce2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
